# Récapitulatif | Un pipeline de classification sur le Titanic

Peut-on estimer la survie d’un passager à partir des informations connues à l’embarquement ? Une ligne représente un passager, la cible vaut 1 s’il a survécu et 0 sinon. C’est une **classification binaire**, sur un événement historique. Un bon score sur ce manifeste ne garantit rien pour un autre naufrage.

**Parcours.** 1. Charger les données / 2. Réserver le test / 3. Explorer / 4. Nettoyer / 5. Construire les variables / 6. Comparer les modèles / 7. Régler les hyperparamètres / 8. Évaluer / 9. Sauvegarder et rejouer.

Ouvrez le notebook dans VS Code avec le noyau Python du dépôt à jour (`uv sync --locked --extra spark`). Les données sont incluses. Exécutez les cellules dans l’ordre. Vous obtiendrez une comparaison de modèles, un bilan sur des billets réservés et un fichier capable de prédire à partir d’une nouvelle ligne brute.

## 1. Charger le manifeste et définir les entrées

Le CSV local reprend les 891 lignes étiquetées du [jeu Titanic](https://www.kaggle.com/competitions/titanic/data). Les noms et cabines ont été retirés et chaque numéro de billet remplacé par un identifiant de groupe. On garde ce dernier pour construire les échantillons.

| Champ | Lecture |
| --- | --- |
| Pclass, Sex, Age | Classe du billet (1 à 3), sexe enregistré, âge en années |
| SibSp, Parch | Frères, sœurs ou conjoints à bord, puis parents ou enfants à bord |
| Fare, Embarked | Tarif enregistré et port (C = Cherbourg, Q = Queenstown, S = Southampton) |
| PassengerId, TicketGroup | Identifiant d’une personne et groupe partageant le billet |
| Survived | Verdict historique, réservé à l’apprentissage et à l’évaluation |

In [ ]:
# Lire le CSV fourni et vérifier qu'une ligne représente bien un passager.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "corrections":
    root = root.parent
data = pd.read_csv(root / "donnees" / "titanic.csv")
inputs = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
assert data.PassengerId.is_unique and set(data.Survived.unique()) == {0, 1}
print(f"{len(data)} passagers, {data.TicketGroup.nunique()} billets distincts.")
display(data[inputs + ["TicketGroup"]].head())

## 2. Réserver le test avant d’explorer les résultats

Deux passagers d’un même billet partagent parfois famille, tarif et conditions de voyage. Si on les répartit au hasard ligne par ligne, la mesure peut être trop favorable. Notre unité de séparation est donc **le billet**. La stratification cherche à garder des proportions de survivants voisines sous cette contrainte.

On isole environ un cinquième du manifeste pour le test final. Dans le reste (le *train*), on prépare quatre plis : dans chaque pli, les billets de validation sont distincts des billets servant à ajuster le modèle. Chaque passager du train sert une fois à la validation. Le test reste à part pendant le choix du modèle.

![Répartition des passagers par usage et plis de validation](figures/recap-titanic-plis.svg)

En vert foncé, les passagers à prédire pour ce pli. En vert clair, les passagers qui servent à ajuster le pipeline. Le nombre sur chaque bande foncée donne son effectif.

In [ ]:
# Fixer un découpage reproductible et contrôler les frontières entre billets.
from sklearn.model_selection import StratifiedGroupKFold

seed = 42


def grouped_folds(frame, n_splits):
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(splitter.split(frame[inputs], frame.Survived, frame.TicketGroup))


dev_idx, test_idx = grouped_folds(data, 5)[0]
train = data.iloc[dev_idx].reset_index(drop=True)
test = data.iloc[test_idx].reset_index(drop=True)
assert set(train.TicketGroup).isdisjoint(test.TicketGroup)
folds = grouped_folds(train, 4)
for a, b in folds:
    assert set(train.iloc[a].TicketGroup).isdisjoint(train.iloc[b].TicketGroup)
X, y = train[inputs], train.Survived
print(f"Développement : {len(train)} personnes. Test réservé : {len(test)}.")

## 3. Explorer les données d’entraînement

Quels champs sont incomplets ? La répartition des âges et la fréquence de survie sont-elles identiques dans toutes les classes de billet ? Regardez à la fois les **effectifs** et les **proportions**. Un taux décrit une association dans ce manifeste, sans isoler un effet causal.

![Âges disponibles ou absents par classe sur le train](figures/recap-titanic-ages.svg)

In [ ]:
# Mesurer les absences puis tracer deux vues complémentaires du train.
profil = pd.DataFrame({"absents": X.isna().sum(), "distincts": X.nunique()})
display(profil)
par_classe = train.groupby("Pclass").Survived.agg(["size", "sum", "mean"])
display(
    par_classe.rename(
        columns={"size": "passagers", "sum": "survivants", "mean": "taux"}
    )
)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), layout="constrained")
axes[0].hist(
    X.Age.dropna(), bins=np.arange(0, 91, 10), color="#317979", edgecolor="white"
)
axes[0].set(
    xlabel="Âge renseigné (ans)", ylabel="Passagers", title="Qui est représenté ?"
)
axes[1].bar(par_classe.index, 100 * par_classe["mean"], color="#317979")
axes[1].set(
    xticks=[1, 2, 3],
    ylim=(0, 100),
    xlabel="Classe du billet",
    ylabel="Survivants (%)",
    title="Survie observée sur le train",
)
plt.show()

**À lire.** L’histogramme compte uniquement les âges renseignés. Dans le deuxième graphique, chaque barre divise le nombre de survivants de la classe par le nombre total de passagers de cette classe. Quelle information perdrait-on en supprimant tous les âges manquants ?

## 4. Data cleaning : apprendre les remplacements dans chaque pli

On contrôle le domaine des mesures et on conserve les âges manquants. Pour chaque colonne numérique, l’imputation remplacera les absences par la médiane **apprise sur les seules lignes d’entraînement du pli**. Les catégories manquantes ont une modalité explicite. On encode les catégories par des indicatrices (one-hot) et on standardise les mesures pour les modèles sensibles aux échelles. Pclass est traitée comme une catégorie.

In [ ]:
# Décrire les opérations sans encore calculer de médiane sur les données.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

assert X.Pclass.isin([1, 2, 3]).all()
assert X[["Age", "Fare", "SibSp", "Parch"]].dropna(how="all").min().ge(0).all()
numeric = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
ports = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="inconnu"),
    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
)
categories = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

## 5. Feature engineering : représenter les proches à bord

Deux colonnes décrivent la famille. Pour un passager avec un conjoint et deux enfants, on obtient **3 proches à bord**. Le calcul additionne SibSp et Parch ; il ne compte pas le passager lui-même. L’arbre peut ainsi séparer les voyageurs seuls des voyageurs accompagnés en un seul test. On garde les deux colonnes originales pour distinguer les compositions familiales.

Le calcul est intégré au pipeline. Une nouvelle ligne aura besoin des mêmes sept champs bruts, et bénéficiera automatiquement de la même préparation.

In [ ]:
# Ajouter le nombre de proches par ligne à la préparation commune.
from sklearn.base import clone

proches = make_pipeline(
    FunctionTransformer(np.asarray),
    FunctionTransformer(np.sum, kw_args={"axis": 1, "keepdims": True}),
    StandardScaler(),
)
prep = ColumnTransformer(
    [
        ("mesures", numeric, ["Age", "Fare", "SibSp", "Parch"]),
        ("port", ports, ["Embarked"]),
        ("classe_sexe", categories, ["Pclass", "Sex"]),
        ("proches", proches, ["SibSp", "Parch"]),
    ]
)
display(X[["SibSp", "Parch"]].head().assign(proches=lambda d: d.SibSp + d.Parch))

## 6. Modeling : quelle valeur ajoutée par rapport à une règle triviale ?

Avec **LazyClassifier**, on essaie sur le premier pli une règle majoritaire, une régression logistique, un arbre, une forêt et un SVM. Notre critère est la **balanced accuracy**, la moyenne de deux proportions : la part des survivants reconnus et la part des non-survivants reconnus. Prédire la classe majoritaire pour tous donne 0,50 si les deux classes sont présentes. L’accuracy ordinaire peut être bien plus élevée.

Ce premier classement sert à explorer : un seul pli peut avantager un candidat. LazyClassifier ajoute sa propre préparation aux colonnes numériques qu’il reçoit. Pour une comparaison maîtrisée, la cellule suivante refait l’évaluation des cinq méthodes avec notre pipeline et les quatre mêmes plis.

La logistique combine les entrées en un score linéaire. L’arbre apprend des conditions successives, la forêt combine plusieurs arbres. Le SVM à noyau permet une frontière courbe entre les deux classes. La règle majoritaire ignore les caractéristiques.

In [ ]:
# Donner à LazyClassifier une validation interne et conserver le test final fermé.
from lazypredict.Supervised import LazyClassifier
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

models = [
    DummyClassifier,
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    SVC,
]
a, b = folds[0]
first_prep = clone(prep).fit(X.iloc[a])
first_train = pd.DataFrame(first_prep.transform(X.iloc[a])).add_prefix("x")
first_val = pd.DataFrame(first_prep.transform(X.iloc[b])).add_prefix("x")
screen = LazyClassifier(classifiers=models, random_state=seed, n_jobs=1)
ranking, _ = screen.fit(
    first_train, first_val, y.iloc[a].to_numpy(), y.iloc[b].to_numpy()
)
assert set(ranking.index) == {cls.__name__ for cls in models}
display(ranking[["Accuracy", "Balanced Accuracy"]])

In [ ]:
# Comparer les mêmes familles sur quatre validations, avec imputation réapprise à chaque fois.
from sklearn.model_selection import cross_val_score


def pipeline(model):
    return Pipeline([("prep", clone(prep)), ("model", model)])


candidates = {
    "Majoritaire": DummyClassifier(strategy="most_frequent"),
    "Logistique": LogisticRegression(max_iter=1000),
    "Arbre": DecisionTreeClassifier(random_state=seed),
    "Forêt": RandomForestClassifier(random_state=seed, n_jobs=1),
    "SVM": SVC(),
}
scores = pd.DataFrame(
    {
        label: cross_val_score(
            pipeline(est), X, y, cv=folds, scoring="balanced_accuracy"
        )
        for label, est in candidates.items()
    },
    index=[f"Pli {i}" for i in range(1, 5)],
).T
scores["moyenne"] = scores.mean(axis=1)
display(scores.sort_values("moyenne", ascending=False).round(3))

## 7. Grid Search ou Random Search à budget égal ?

Un **hyperparamètre** est un réglage choisi avant l’apprentissage. Pour isoler l’effet de la recherche, on prend la même famille (la forêt), les mêmes plis, le même critère et six configurations par méthode. La grille croise trois profondeurs et deux tailles minimales de feuille. La recherche aléatoire tire six couples distincts dans un espace plus fin. Une feuille plus grande limite les décisions fondées sur quelques passagers.

Chaque méthode ajuste 6 × 4 = 24 forêts pour la validation. L’étendue des espaces diffère volontairement. Le gagnant décrit cette expérience et cette graine. Pour le choix final, les autres modèles du comparatif restent en lice.

In [ ]:
# Régler la même forêt avec un budget de six configurations par recherche.
from time import perf_counter

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

settings = {
    "estimator": pipeline(candidates["Forêt"]),
    "cv": folds,
    "scoring": "balanced_accuracy",
    "n_jobs": 1,
    "refit": False,
    "error_score": "raise",
}
searches = {
    "Grille": GridSearchCV(
        param_grid={
            "model__max_depth": [3, 6, None],
            "model__min_samples_leaf": [1, 5],
        },
        **settings,
    ),
    "Aléatoire": RandomizedSearchCV(
        param_distributions={
            "model__max_depth": [2, 3, 4, 5, 6, 8, 10, None],
            "model__min_samples_leaf": [1, 2, 3, 5, 8, 12],
        },
        n_iter=6,
        random_state=seed,
        **settings,
    ),
}
times = {}
for label, search in searches.items():
    start = perf_counter()
    search.fit(X, y)
    times[label] = perf_counter() - start
    candidates[label] = clone(candidates["Forêt"]).set_params(
        **{
            key.removeprefix("model__"): value
            for key, value in search.best_params_.items()
        }
    )
summary = pd.DataFrame(
    [
        {
            "recherche": label,
            "essais": len(s.cv_results_["params"]),
            "moyenne CV": s.best_score_,
            "paramètres": s.best_params_,
            "secondes sur ce poste": round(times[label], 2),
        }
        for label, s in searches.items()
    ]
)
display(summary)
for label, search in searches.items():
    plt.scatter(
        [label] * len(search.cv_results_["params"]),
        search.cv_results_["mean_test_score"],
        alpha=0.7,
    )
plt.ylabel("Balanced accuracy moyenne sur les 4 plis")
plt.title("Un point par configuration de forêt")
plt.show()

## 8. Évaluer une seule fois sur les billets réservés

On choisit la meilleure moyenne de validation parmi les modèles initiaux et les deux forêts réglées. En cas d’égalité, on conserve le premier candidat dans l’ordre du tableau de définition. On le réentraîne sur tout le train puis on ouvre le test. Les prédictions suivent la règle par défaut du classifieur. Modifier le modèle après avoir vu ce bilan demanderait un nouveau test indépendant.

In [ ]:
# Figer le choix avant d'utiliser la moindre étiquette de test.
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)

selection = scores["moyenne"].to_dict()
selection.update({label: search.best_score_ for label, search in searches.items()})
winner = max(selection, key=selection.get)
fitted = pipeline(candidates[winner]).fit(X, y)
pred = fitted.predict(test[inputs])
print(
    f"Choix sur validation : {winner}. Balanced accuracy CV : {selection[winner]:.3f}"
)
print(
    f"Test : accuracy {accuracy_score(test.Survived, pred):.3f}, balanced accuracy {balanced_accuracy_score(test.Survived, pred):.3f}"
)
print(
    classification_report(
        test.Survived,
        pred,
        target_names=["Non-survivant", "Survivant"],
        digits=3,
        zero_division=0,
    )
)
ConfusionMatrixDisplay.from_predictions(
    test.Survived,
    pred,
    display_labels=["Non-survivant", "Survivant"],
    cmap="Blues",
    colorbar=False,
)
plt.xlabel("Classe prédite")
plt.ylabel("Classe observée")
plt.title("Chaque passager du test apparaît dans une seule case")
plt.show()

**Lecture.** Sur la ligne « Survivant », la case de gauche compte les survivants classés à tort parmi les non-survivants. Le *rappel* divise les survivants reconnus par tous les survivants observés ; la *précision* les divise par toutes les personnes prédites survivantes. Le faible effectif et le voyage unique limitent la généralisation. Comme chez Nervure, le score de sélection et le bilan réservé répondent à deux questions différentes.

## 9. Sauvegarder, recharger et prédire sur un manifeste brut

On conserve dans un même fichier les transformations apprises, les colonnes attendues et le modèle. Les versions des bibliothèques et la graine facilitent la reproduction. Le nom dépend du contenu, ce qui permet de distinguer les versions. Rechargez uniquement vos propres fichiers joblib avec l’environnement verrouillé du dépôt.

In [ ]:
# Rejouer les décisions avec le fichier sauvegardé, puis traiter un nouvel exemple.
import hashlib
from importlib.metadata import version
from io import BytesIO

import joblib

bundle = {
    "pipeline": fitted,
    "inputs": inputs,
    "modèle": winner,
    "seed": seed,
    "cible": "Survived (1 = survie)",
    "versions": {p: version(p) for p in ["scikit-learn", "numpy", "pandas", "joblib"]},
}
folder = root / "artefacts" / "recap_titanic"
folder.mkdir(exist_ok=True, parents=True)
buffer = BytesIO()
joblib.dump(bundle, buffer)
payload = buffer.getvalue()
path = folder / f"pipeline-{hashlib.sha256(payload).hexdigest()[:16]}.joblib"
path.write_bytes(payload)
restored = joblib.load(path)
np.testing.assert_array_equal(
    restored["pipeline"].predict(test[restored["inputs"]]), pred
)
example = pd.DataFrame([[3, "male", np.nan, 1, 0, 12.5, "S"]], columns=inputs)
print("Fichier :", path.name)
print(
    "Classe prédite du passager fictif (0 ou 1) :",
    restored["pipeline"].predict(example)[0],
)
print(f"Rejeu vérifié sur {len(pred)} passagers, sans nouvel entraînement.")

## Défis facultatifs

1. **Quantifier la baseline.** Sur les quatre plis, quelle accuracy ordinaire obtient la règle majoritaire ? Expliquez pourquoi sa balanced accuracy reste à 0,50. Partez de `pipeline(candidates["Majoritaire"])` et de `cross_val_score(..., cv=folds, scoring="accuracy")`.
2. **Tester l’apport des proches.** Avec la même régression logistique et les mêmes plis, retirez uniquement la branche `proches` du prétraitement. Partez de `pipeline(candidates["Logistique"]).set_params(prep__proches="drop")` et comparez les écarts pli par pli. Pourquoi un indicateur construit comme une somme peut-il apporter peu à un modèle linéaire ?

[Solutions expliquées et résultats](corrections/Recap_pipeline_ML_Titanic.ipynb)

**Pour approfondir** : [Pipelines scikit-learn](https://scikit-learn.org/stable/modules/compose.html), [recherche d’hyperparamètres](https://scikit-learn.org/stable/modules/grid_search.html) et [Lazy Predict](https://github.com/shankarpandala/lazypredict).